We are going to make use of an homography in order to start detecting players in a given image

# 1. Import necessary libraries

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
%matplotlib widget
%matplotlib inline

# 2. Define global variables

In [ ]:
INPUT_RANDOM_IMG = "../data/raw/random_img.jpg"
INPUT_FOOTBALL_IMG = "../data/raw/soccernet/tracking/test/SNMOT-124/img1/000001.jpg"

In [ ]:
PENALTY_AREA_DEPTH = 16.5
PENALTY_AREA_WIDTH = 40.32
LITTLE_PENALTY_AREA_DEPTH = 5.5
LITTLE_PENALTY_AREA_WIDTH = 18.32
CENTER_CIRCLE_RADIUS = 9.15

# 3. Functions

# 4. Code

## 4.1. Random image

In [ ]:
random_img = cv2.imread(INPUT_RANDOM_IMG)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(random_img, cv2.COLOR_BGR2RGB))
plt.grid(True)
plt.show()

We need to highlight the four points we are oging to use in order to perform the homography

In [ ]:
src = np.float32([[210, 110], [700, 245], [430, 350], [0, 220]])

In [ ]:
SIDE = 400
dest = np.float32([[0, 0], [SIDE, 0], [SIDE, SIDE], [0, SIDE]])

In [ ]:
H = cv2.getPerspectiveTransform(src, dest)

In [ ]:
H

In [ ]:
warped = cv2.warpPerspective(random_img, H, (SIDE, SIDE))
warped

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
plt.grid(True)
plt.show()

## 4.2. Pitch image

### 4.2.1. Create the empty pitch

In [ ]:
def m2px(x_m, y_m, scale):
    """
    Convert meters to pixels.
    """
    x_px = int(x_m * scale)
    y_px = int(y_m * scale)
    return x_px, y_px

In [ ]:
def draw_pitch(
    length: float = 105, width: float = 68, scale: float = 10, margin: float = 3
) -> np.ndarray:
    """
    Empty football pitch drawn from vertical view. scale = pixels per meter.
    Scale = 10 represents 10 pixels per meter, so a 105x68 pitch will be 1050x680 pixels.

    Parameters:
        - length: Length of the pitch in meters (default is 105).
        - width: Width of the pitch in meters (default is 68).
        - scale: Scale factor for the pitch (default is 10).
        - margin: Margin around the pitch in meters (default is 3).
    Returns:
        - pitch: A numpy array representing the empty football pitch.
    """

    pitch_length = int(length * scale)
    pitch_width = int(width * scale)

    pitch = np.zeros((pitch_width, pitch_length, 3), dtype=np.uint8)
    pitch[:] = (30, 90, 30)  # Green background

    # Draw the pitch boundaries
    cv2.rectangle(pitch, (0, 0), (pitch_length - 1, pitch_width - 1), (255, 255, 255), 2)

    # Draw the center line
    cv2.line(
        pitch, (pitch_length // 2, 0), (pitch_length // 2, pitch_width - 1), (255, 255, 255), 2
    )

    # Draw penalty areas
    for pen_width, pen_depth in [
        (PENALTY_AREA_WIDTH, PENALTY_AREA_DEPTH),
        (LITTLE_PENALTY_AREA_WIDTH, LITTLE_PENALTY_AREA_DEPTH),
    ]:
        cv2.rectangle(
            pitch,
            m2px(0, (width - pen_width) / 2, scale),
            m2px(pen_depth, (width + pen_width) / 2, scale),
            (255, 255, 255),
            2,
        )  # Left penalty areas
        cv2.rectangle(
            pitch,
            m2px(length - pen_depth, (width - pen_width) / 2, scale),
            m2px(length, (width + pen_width) / 2, scale),
            (255, 255, 255),
            2,
        )  # Right penalty areas

    # Center circle and center point
    cv2.circle(
        pitch,
        (pitch_length // 2, pitch_width // 2),
        int(CENTER_CIRCLE_RADIUS * scale),
        (255, 255, 255),
        2,
    )  # Center circle
    cv2.circle(pitch, (pitch_length // 2, pitch_width // 2), 3, (255, 255, 255), -1)  # Center point

    return pitch

In [ ]:
empty_pitch = draw_pitch()
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(empty_pitch, cv2.COLOR_BGR2RGB))
plt.grid(True)

### 4.2.2. Load the football image

In [ ]:
football_img = cv2.imread(INPUT_FOOTBALL_IMG)

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(football_img, cv2.COLOR_BGR2RGB))
plt.grid(
    True,
)
plt.show()

In [ ]:
IMG_PTS = {
    "center_circle": (978, 580),
    "circle_top": (978, 470),
    "circle_bottom": (978, 750),
    "circle_left": (245, 600),
    "circle_right": (1690, 600),
    "middle_background": (970, 290),  # línea de medio campo contra la banda lejana
}

In [ ]:
vis = football_img.copy()
for name, (x, y) in IMG_PTS.items():
    cv2.circle(vis, (int(x), int(y)), 8, (0, 0, 255), -1)
    cv2.putText(vis, name, (int(x) + 12, int(y)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))

In [ ]:
SCALE = 10

PITCH_PTS = {
    "center_circle": (52.5, 34.0),
    "circle_top": (52.5, 24.85),
    "circle_bottom": (52.5, 43.15),
    "circle_left": (43.35, 34.0),
    "circle_right": (61.65, 34.0),
    "middle_background": (52.5, 0.0),
}

In [ ]:
names = list(IMG_PTS.keys())  # mismo orden en ambos
# Define the source and the destination points for the homography
src = np.float32([IMG_PTS[n] for n in names])
dst = np.float32([PITCH_PTS[n] for n in names]) * SCALE

In [ ]:
H, mask = cv2.findHomography(src, dst, 0)

In [ ]:
# Transform the source points using the homography matrix
proj = cv2.perspectiveTransform(src.reshape(-1, 1, 2), H).reshape(-1, 2)
# Calculate the reprojection error
err = np.linalg.norm(proj - dst, axis=1) / SCALE
for n, e in zip(names, err, strict=False):
    print(f"{n:16s} {e:5.2f} m")

In [ ]:
# If we draw the homography on the pitch, we can see how well it aligns with the actual points
# on the pitch.
pitch = draw_pitch(scale=SCALE)
h, w = pitch.shape[:2]

# Perform the perspective warp of the football image using the homography matrix
warped = cv2.warpPerspective(football_img, H, (w, h))
# Add the warped image and the pitch together to create an overlay
overlay = cv2.addWeighted(warped, 0.7, pitch, 0.6, 0)

plt.figure(figsize=(14, 9))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))

In [ ]:
# save the homography and points to a JSON file
out = Path("../data/interim/homography_SNMOT-124.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(
    json.dumps(
        {
            "sequence": "SNMOT-124",
            "frame": 1,
            "scale": SCALE,
            "image_points": IMG_PTS,
            "pitch_points": PITCH_PTS,
            "H": H.tolist(),
        },
        indent=2,
    )
)